In [ ]:
import os

PROJECT_ROOT = "/Users/zanderholleran/Desktop/python_projects/mesa_lcc_model"
os.chdir(PROJECT_ROOT)

%pwd



# Enable auto-reloading of custom modules
%load_ext autoreload
%autoreload 2

In [ ]:
from season.persons import SeasonPerson
from season.configs import ScheduleSpecs, SeasonConfig, DayParams, make_season_config, PopulationParams
from season.season_orchestrator import SeasonOrchestrator

import numpy as np
import pandas as pd
from scipy.stats import norm, lognorm, skewnorm, truncnorm, uniform

# Mesa model components
import mesa
from mesa import model
from mesa import agent
from traffic.model.traffic_model import TrafficModel
from traffic.agents.vehicle_agent import VehicleAgent
from traffic.agents.road_segment_agent import RoadSegmentAgent

import traffic.utils.unit_conversion_utils as uc
import traffic.utils.analysis_utils as au
import traffic.utils.animation_utils as anim

# Visualization and display (Jupyter-specific)
from IPython.display import display, HTML
pd.set_option("display.max_rows", None)  

# Define the config

In [ ]:
pop_params = PopulationParams(
    population_size=2000,
    prior_car=22.0,
    prior_bus=40.0,

)


config = make_season_config(
    season_id='one_day',
    run_description='One day test run',
    seed=124,
    n_days=1,
    max_steps=2000,
    max_persons=2000,
    collect_every_n=10,
    batch_run=False, 

    start_hr=8,
    bus_capacity=30,
    road_path='data/roads/hw210_sl_and_curvs.parquet',
    ecs_path='data/vehicle_counts/expected_counts_seconds.csv',
    toll_mechanism='static',
    toll_params={'car': 8.0, 'bus': 0.0},
   
    traffic_percentile_schedule=ScheduleSpecs(mode='static', value=80),  # static bus interval of 15 minutes,
    bus_interval_schedule=ScheduleSpecs(mode='static', value=30),
    crashes_schedule=ScheduleSpecs(mode='static', value=4), 
    population_params=pop_params,
    canyon_closures_schedule=None,
)

# Run single day

In [ ]:
# Example usage of SeasonOrchestrator with example_config
orchestrator = SeasonOrchestrator(season_config=config)
tm = orchestrator.run_day()

# Analysis

In [ ]:
finished_agents = au.finished_agents_summary_df(tm, plots=True)


In [ ]:
# process the finished_agents data 
vehicles_full = au.vehicle_agent_data_time_series(tm, plots=True)
model_ts = tm.datacollector.get_model_vars_dataframe()


au.plot_speed_delta(vehicles_full, model_ts)

# Annimations

In [ ]:
# run the animation
# looking at one car
issue_car_id =  604
issue_step = 0

In [ ]:
anim.animate_traffic(vehicles_full, road_gdf, interval=100, step_skip=2, watch=None, zoom=20)

In [ ]:
anim.animate_traffic_with_speed_delta_highlight(vehicles_full, road_gdf, model_ts, interval=100, step_skip=3, watch=None, zoom=20)

In [ ]:
anim.animate_relative_distance(vehicle_df=vehicles_full, agent_id=700, distance_behind=100, color_by='status')